In [ ]:
!pip install langchain-huggingface[full] sentence-transformers

In [ ]:
!pip install langchain-pinecone pinecone-client

In [ ]:
!pip install langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [ ]:
!pip install docling

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
import io
from PIL import Image
from tqdm import tqdm

def generate_image_description(image: Image.Image) -> str:
    """Mock function: Pass the PIL Image to your Vision Model."""
    return "[IMAGE CONTEXT: A screenshot showing the main dashboard.]"

def upload_to_minio(image: Image.Image, image_filename: str) -> str:
    """Mock function: Upload image to MinIO bucket."""
    return f"https://your-minio-server/app_images/{image_filename}"

def parse_with_docling_light(pdf_path: str):
    """
    Lightweight Docling parser: Uses nearest-neighbor text injection to prevent Pydantic schema crashes.
    """
    print(f"📄 Phase 1: Docling is analyzing PDF layout for: {pdf_path}")
    print("⏳ Please wait, mapping document structure (OCR Disabled for speed)...")

    # 1. Configure Docling
    pipeline_options = PdfPipelineOptions()
    pipeline_options.generate_picture_images = True
    pipeline_options.do_ocr = False  # Keep OCR off for speed

    converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    # 2. Process document
    result = converter.convert(pdf_path)
    document = result.document
    document_elements = list(document.iterate_items())

    print("\n🚀 Phase 2: Processing Images & Generating Vision Context")

    img_counter = 0
    last_text_element = None
    pending_descriptions = []

    # 3. Iterate through elements
    for element, level in tqdm(document_elements, desc="Parsing Elements", unit=" block", dynamic_ncols=True):

        # A. Track the most recent text element (paragraph, header, etc.)
        if hasattr(element, "text") and isinstance(element.text, str):
            last_text_element = element

            # If we found an image at the very top of the page before any text, inject it here
            if pending_descriptions:
                element.text = "".join(pending_descriptions) + "\n" + element.text
                pending_descriptions.clear()

        # B. When we hit a picture, process it and attach the text to our tracked element
        elif element.label == "picture":
            image_obj = element.get_image(document)
            if image_obj:
                image_filename = f"docling_img_{img_counter}.png"
                img_counter += 1

                # Process the image
                image_url = upload_to_minio(image_obj, image_filename)
                description = generate_image_description(image_obj)

                formatted_desc = f"\n\n> 🖼️ **[Image Reference: {image_url}]**\n> {description}\n"

                # Inject it seamlessly into the nearest text block!
                if last_text_element:
                    last_text_element.text += formatted_desc
                else:
                    pending_descriptions.append(formatted_desc)

    print("\n📝 Phase 3: Exporting to Markdown")
    # 4. Export the final document. The exporter will naturally include our injected descriptions!
    final_markdown = document.export_to_markdown()

    print("✅ Full parsing and enrichment complete!")
    return final_markdown



In [ ]:
# --- To run the parser ---
enriched_markdown = parse_with_docling_light("/content/final_documentation new5.pdf")
print(enriched_markdown)

📄 Phase 1: Docling is analyzing PDF layout for: /content/final_documentation new5.pdf
⏳ Please wait, mapping document structure (OCR Disabled for speed)...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]


🚀 Phase 2: Processing Images & Generating Vision Context


Parsing Elements: 100%|██████████| 1459/1459 [00:00<00:00, 47165.51 block/s]


📝 Phase 3: Exporting to Markdown


✅ Full parsing and enrichment complete!
<!-- image -->



&gt; 🖼️ **[Image Reference: https://your-minio-server/app\_images/docling\_img\_0.png]**
&gt; [IMAGE CONTEXT: A screenshot showing the main dashboard.]

Page | 1

| Chapter 1: Introduction ..........................................................................................................   | Chapter 1: Introduction ..........................................................................................................   |
|--------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------|
| 1.1 Overview                                                                                                                         | ..................................................................................................

# store the meta data by using langchain

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_markdown_for_pinecone(final_markdown: str, source_file: str, category: str):
    # 1. Define the headers we want LangChain to look for
    headers_to_split_on = [
        ("#", "Section"),        # Catches # Title
        ("##", "Sub-Section"),   # Catches ## ➢ High Knees
        ("###", "Topic")         # Catches ### Sub-topics if any exist
    ]

    # 2. Initialize the Markdown Splitter
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    # 3. Split the text based on the headers
    md_header_splits = markdown_splitter.split_text(final_markdown)

    # 4. Size limit enforcement
    chunk_size = 1000
    chunk_overlap = 150
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    final_chunks = text_splitter.split_documents(md_header_splits)

    # 5. THE ENHANCEMENT: Inject global metadata into every chunk
    for chunk in final_chunks:
        # LangChain keeps the headers, we just add the global tags
        chunk.metadata["source"] = source_file
        chunk.metadata["category"] = category

    print(f"✅ Created {len(final_chunks)} perfectly formatted chunks.")
    return final_chunks

# --- How to run it ---
chunks = chunk_markdown_for_pinecone(
    final_markdown=enriched_markdown,
    source_file="Sports_AI_Assistant_Catalogue.pdf",
    category="App Guide"
)

NameError: name 'enriched_markdown' is not defined

# store the meta data to drive

# can be ignored but connect with t4 first

In [ ]:
import pickle
from google.colab import drive

# 1. Mount Google Drive to the Colab notebook
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define the exact folder and filename in your Drive
# This saves it right in the main directory of your Google Drive
SAVE_PATH = '/content/drive/MyDrive/doc_guid_metadata_app_manual_chunks_backup.pkl'

# 3. Save (dump) the LangChain chunks to the pickle file
print(f"Saving data to {SAVE_PATH}...")
with open(SAVE_PATH, 'wb') as file:
    pickle.dump(chunks, file)

print(f"✅ Backup Complete: Successfully saved {len(chunks)} chunks to Google Drive!")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving data to /content/drive/MyDrive/doc_guid_metadata_app_manual_chunks_backup.pkl...


NameError: name 'chunks' is not defined

In [ ]:
import pickle
from google.colab import drive

# 1. Mount Google Drive (Required if this is a brand new session)
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Point to the file you saved earlier
LOAD_PATH = '/content/drive/MyDrive/app_manual_chunks_backup.pkl'

# 3. Load the data back into a Python variable
print(f"Loading data from {LOAD_PATH}...")
with open(LOAD_PATH, 'rb') as file:
    restored_chunks = pickle.load(file)

print(f"✅ Restore Complete: Successfully loaded {len(restored_chunks)} chunks into memory!")

# --- Now you can run your Pinecone upload! ---
# vectorstore = upload_and_verify(restored_chunks)

In [ ]:
# 1. Define your new, clean metadata labels
NEW_CATEGORY = "assistant_user_guide"
NEW_SOURCE_NAME = "Sports_AI_Assistant_Catalogue.pdf"

print("🔄 Updating metadata in all chunks...")

# 2. Loop through the chunks and update the dictionary
for chunk in chunks:
    chunk.metadata['category'] = NEW_CATEGORY
    chunk.metadata['source'] = NEW_SOURCE_NAME

print(f"✅ Successfully updated metadata for {len(chunks)} chunks!")

# upload the metadata textchunk to pinecone

In [ ]:
import os
os.environ['PINECONE_API_KEY'] = '######'

In [ ]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

# --- 1. CONFIGURATION ---
# Replace with your actual Pinecone API key
# os.environ['PINECONE_API_KEY'] = 'your-pinecone-api-key'

PINECONE_INDEX_NAME = "medical-sport-assistant"
TARGET_NAMESPACE = "assistant_user_guide"

# --- 2. INITIALIZE BGE LARGE EMBEDDINGS ---
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {'device': 'cuda'} # Change to 'cpu' if not running on Colab/GPU
encode_kwargs = {'normalize_embeddings': True}

print("🧠 Loading BGE-Large Embedding Model...")
# ✅ Changed to match the class you imported
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

def upload_and_verify(chunks):
    print(f"\n🚀 Starting upload of {len(chunks)} chunks to index '{PINECONE_INDEX_NAME}' under namespace '{TARGET_NAMESPACE}'...")

    # --- 3. THE UPLOAD (UPSERT) ---
    vectorstore = PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        index_name=PINECONE_INDEX_NAME,
        namespace=TARGET_NAMESPACE
    )
    print("✅ Upload complete!")

    # --- 4. THE ANALYSIS (VERIFICATION) ---
    print("\n📊 --- PINECONE INDEX ANALYSIS ---")

    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
    index = pc.Index(PINECONE_INDEX_NAME)

    # Fetch live stats from your medical-sport-assistant index
    stats = index.describe_index_stats()

    print(f"Total Vectors in Entire Index: {stats.total_vector_count}")
    print(f"Index Dimensionality: {stats.dimension} (Should match BGE's 1024)")

    namespaces = stats.namespaces
    if TARGET_NAMESPACE in namespaces:
        uploaded_count = namespaces[TARGET_NAMESPACE].vector_count
        print(f"✅ SUCCESS: Namespace '{TARGET_NAMESPACE}' is active with {uploaded_count} vectors.")

        if uploaded_count == len(chunks):
            print("🎯 Perfect Match! Every chunk was successfully embedded and uploaded.")
        else:
            print(f"⚠️ Note: You uploaded {len(chunks)} chunks, but the namespace has {uploaded_count} vectors. (Normal if appending to existing data).")
    else:
        print(f"❌ ERROR: Namespace '{TARGET_NAMESPACE}' not found.")

    print("---------------------------------")
    return vectorstore

vectorstore = upload_and_verify(chunks)